# Module 5: LLM Deployment

**Goal:** Build and test a production-ready LLM API with caching, rate limiting, and observability.

**What you'll learn:**
1. Deployment options: cloud APIs vs. self-hosting trade-offs
2. Building a FastAPI wrapper with streaming support
3. Semantic caching to cut costs 40-60%
4. Rate limiting and cost tracking
5. Health checks and graceful error handling
6. Load testing your API

---

## 0. Setup

In [ ]:
%pip install -q openai fastapi uvicorn[standard] httpx pydantic python-dotenv

In [ ]:
import os, json, time, hashlib, asyncio
from dotenv import load_dotenv
load_dotenv(dotenv_path="../.env")

from openai import AsyncOpenAI, OpenAI
client = OpenAI()
async_client = AsyncOpenAI()
print("✅ Ready")

---
## 1. Deployment Options: The Trade-off Matrix

| Option | Latency | Cost | Privacy | Complexity |
|--------|---------|------|---------|------------|
| **Cloud API** (OpenAI, Anthropic) | ~500ms | Pay-per-token | Data sent to provider | Low |
| **Self-hosted** (vLLM + open model) | ~100ms | Fixed infra cost | Full control | High |
| **Hybrid router** | Varies | Optimised | Mixed | Medium |

**Rule of thumb:** Start with cloud API. Switch to self-hosted when monthly API cost > $2k/month or data privacy is non-negotiable.

---
## 2. Semantic Cache

The biggest single latency and cost reduction: cache semantically similar queries. "What's the capital of France?" and "Tell me the capital city of France" should both hit the cache.

In [ ]:
from typing import Optional
import numpy as np

class SemanticCache:
    """In-memory semantic cache using cosine similarity on embeddings."""

    def __init__(self, similarity_threshold: float = 0.95, max_entries: int = 1000):
        self.threshold = similarity_threshold
        self.max_entries = max_entries
        self.entries: list[dict] = []   # [{embedding, response, query, hits}]
        self.stats = {"hits": 0, "misses": 0}

    def _embed(self, text: str) -> list[float]:
        response = client.embeddings.create(model="text-embedding-3-small", input=text)
        return response.data[0].embedding

    def _cosine_similarity(self, a: list, b: list) -> float:
        a, b = np.array(a), np.array(b)
        return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

    def get(self, query: str) -> Optional[str]:
        embedding = self._embed(query)
        for entry in self.entries:
            sim = self._cosine_similarity(embedding, entry["embedding"])
            if sim >= self.threshold:
                entry["hits"] += 1
                self.stats["hits"] += 1
                return entry["response"]
        self.stats["misses"] += 1
        return None

    def set(self, query: str, response: str):
        if len(self.entries) >= self.max_entries:
            # Evict least-hit entry
            self.entries.sort(key=lambda x: x["hits"])
            self.entries.pop(0)
        self.entries.append({
            "query": query,
            "response": response,
            "embedding": self._embed(query),
            "hits": 0,
        })

    @property
    def hit_rate(self) -> float:
        total = self.stats["hits"] + self.stats["misses"]
        return self.stats["hits"] / total if total else 0.0


cache = SemanticCache(similarity_threshold=0.92)

def cached_completion(query: str) -> tuple[str, bool]:
    """Returns (response, was_cached)."""
    cached = cache.get(query)
    if cached:
        return cached, True
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": query}],
        temperature=0,
    ).choices[0].message.content
    cache.set(query, response)
    return response, False

# Warm the cache
queries = [
    "What is the capital of France?",
    "Tell me the capital city of France",      # Should hit cache
    "What's France's capital?",                # Should hit cache
    "What is the capital of Germany?",         # Should miss cache
]

for q in queries:
    start = time.time()
    resp, cached = cached_completion(q)
    elapsed = (time.time() - start) * 1000
    source = "CACHE" if cached else "API  "
    print(f"[{source}] {elapsed:6.0f}ms  Q: {q[:45]:45s}  A: {resp[:30]}")

print(f"\nCache hit rate: {cache.hit_rate:.0%}")

---
## 3. Streaming Responses

For interactive UIs: stream tokens as they're generated instead of waiting for the full response. Reduces **perceived** latency dramatically.

In [ ]:
import sys

def stream_response(prompt: str, model="gpt-4o-mini"):
    """Stream tokens to stdout as they arrive."""
    first_token_time = None
    start = time.time()
    full_response = ""

    stream = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        stream=True,
    )

    for chunk in stream:
        token = chunk.choices[0].delta.content or ""
        if token:
            if first_token_time is None:
                first_token_time = time.time()
                ttft = (first_token_time - start) * 1000
                print(f"[TTFT: {ttft:.0f}ms] ", end="", flush=True)
            print(token, end="", flush=True)
            full_response += token

    total_ms = (time.time() - start) * 1000
    print(f"\n\n[Total: {total_ms:.0f}ms | {len(full_response)} chars]")
    return full_response

stream_response("List 5 best practices for REST API design, one line each.")

---
## 4. Cost Tracking

Token costs add up fast. Track them per request from day one.

In [ ]:
# GPT-4o-mini pricing (June 2025) — update when pricing changes
PRICING = {
    "gpt-4o-mini":   {"input": 0.15 / 1_000_000,  "output": 0.60 / 1_000_000},
    "gpt-4o":        {"input": 2.50 / 1_000_000,  "output": 10.0 / 1_000_000},
    "gpt-4-turbo":   {"input": 10.0 / 1_000_000,  "output": 30.0 / 1_000_000},
}

def call_with_cost_tracking(prompt: str, model="gpt-4o-mini") -> dict:
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    usage = response.usage
    pricing = PRICING.get(model, PRICING["gpt-4o-mini"])
    cost = usage.prompt_tokens * pricing["input"] + usage.completion_tokens * pricing["output"]

    return {
        "content": response.choices[0].message.content,
        "input_tokens": usage.prompt_tokens,
        "output_tokens": usage.completion_tokens,
        "cost_usd": cost,
        "model": model,
    }

# Compare costs across models for the same prompt
prompt = "Explain the CAP theorem in 3 sentences."
for model in ["gpt-4o-mini", "gpt-4o"]:
    result = call_with_cost_tracking(prompt, model=model)
    print(f"{model:20s}  tokens={result['input_tokens']+result['output_tokens']:4d}  cost=${result['cost_usd']:.6f}")

print("\nAt 1M calls/day:")
for model in ["gpt-4o-mini", "gpt-4o"]:
    result = call_with_cost_tracking(prompt, model=model)
    daily = result["cost_usd"] * 1_000_000
    print(f"  {model:20s}  ${daily:,.0f}/day  (${daily*30:,.0f}/month)")

---
## 5. FastAPI Service (run as a script)

The cell below writes a complete FastAPI app to disk and shows how to run it.

In [ ]:
api_code = '''
# Run with: uvicorn api_server:app --reload --port 8000
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from openai import OpenAI
from dotenv import load_dotenv
import time, os

load_dotenv()
app = FastAPI(title="LLM API", version="1.0")
client = OpenAI()
request_log = []   # In production use a real DB

class ChatRequest(BaseModel):
    message: str
    model: str = "gpt-4o-mini"
    max_tokens: int = 512

@app.get("/health")
def health():
    return {"status": "ok", "requests_served": len(request_log)}

@app.post("/chat")
def chat(req: ChatRequest):
    start = time.time()
    try:
        response = client.chat.completions.create(
            model=req.model,
            messages=[{"role": "user", "content": req.message}],
            max_tokens=req.max_tokens,
        )
        content = response.choices[0].message.content
        latency_ms = (time.time() - start) * 1000
        request_log.append({"latency_ms": latency_ms, "tokens": response.usage.total_tokens})
        return {"response": content, "latency_ms": round(latency_ms), "tokens": response.usage.total_tokens}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))
'''

with open("api_server.py", "w") as f:
    f.write(api_code.strip())

print("✅ api_server.py written")
print("\nTo run it:")
print("  uvicorn api_server:app --reload --port 8000")
print("\nThen test it:")
print('  curl -X POST http://localhost:8000/chat -H "Content-Type: application/json" -d \'{"message": "Hello"}\'')

---
## 6. Testing Your API with httpx

In [ ]:
# Run this after starting the server above: uvicorn api_server:app --port 8000
# (Start it in a terminal, then run this cell)

import httpx

BASE_URL = "http://localhost:8000"

def test_api():
    # Health check
    health = httpx.get(f"{BASE_URL}/health")
    print(f"Health: {health.json()}")

    # Chat
    test_messages = [
        "What is 2+2?",
        "Name the planets in order from the sun.",
    ]
    for msg in test_messages:
        resp = httpx.post(f"{BASE_URL}/chat", json={"message": msg}, timeout=30)
        data = resp.json()
        print(f"Q: {msg[:30]:30s}  → {data['response'][:50]}...  ({data['latency_ms']}ms)")

# Uncomment once the server is running:
# test_api()
print("Start the server first (see cell above), then uncomment test_api()")

---
## 7. Production Deployment Checklist

```
Infrastructure
  ✅ Health endpoint (/health)
  ✅ Graceful error handling (no raw tracebacks to clients)
  ✅ Request/response logging
  ✅ CORS configured for your frontend domain
  □  Rate limiting (per-user, not per-IP)
  □  Authentication (API keys or JWT)

Reliability
  ✅ Timeout on LLM calls (never wait forever)
  □  Retry with exponential backoff on 429/500
  □  Fallback model (GPT-4o → GPT-4o-mini on error)
  □  Circuit breaker

Cost
  ✅ Semantic cache
  ✅ Cost tracking per request
  □  Budget alerts (Slack/email when $X/day exceeded)
  □  Model router (cheap model for simple queries)
```

## 🧪 Exercises

1. **Measure cache savings**: Run 20 queries (10 unique, each asked twice). What's the actual latency and cost savings vs no cache?
2. **Add retry logic**: Wrap `cached_completion` with exponential backoff using `tenacity`. Test it by temporarily using a bad API key.
3. **Model router**: Add logic to route simple queries (< 20 words, no code) to `gpt-4o-mini` and complex ones to `gpt-4o`. Measure cost difference.
4. **Load test**: Use `asyncio.gather` to send 20 concurrent requests to your FastAPI server. What's the p95 latency?

---
**Next:** [Module 06 — Optimization](../06-optimization/optimization.ipynb)